### 파생 변수 생성 (실행 전 GJR-GARCH,1-step.ipynb 실행할 것)

In [9]:
import pandas as pd
import numpy as np

# =========================
# 1. 인덱스 정렬
# =========================
data_2008=pd.read_csv(r'..\..\data\raw\data_2008.csv', index_col=0, parse_dates=True)
data_2009=pd.read_csv(r'..\..\data\raw\data_2009.csv', index_col=0, parse_dates=True)

data_2008 = data_2008.sort_index()
data_2009 = data_2009.sort_index()

# =========================
# 2. 기준 컬럼
# =========================
price_col = 'KOSPI 200 Close'

# =========================
# 3. MA / EMA 생성
# =========================
data_2008['KOSPI 200_MA5'] = data_2008[price_col].rolling(window=5).mean()
data_2008['KOSPI 200_MA15'] = data_2008[price_col].rolling(window=15).mean()

data_2008['KOSPI 200_EMA5'] = data_2008[price_col].ewm(span=5, adjust=False).mean()
data_2008['KOSPI 200_EMA15'] = data_2008[price_col].ewm(span=15, adjust=False).mean()

# =========================
# 4. RSI(14) 생성
# =========================
delta = data_2008[price_col].diff()

gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)

# 단순이동평균 방식 RSI
avg_gain = gain.rolling(window=14).mean()
avg_loss = loss.rolling(window=14).mean()

rs = avg_gain / avg_loss
data_2008['KOSPI 200_RSI14'] = 100 - (100 / (1 + rs))

# 만약 loss가 0이면 RSI가 100에 가까워질 수 있음
# 필요시 아래처럼 무한대 처리 가능
data_2008['KOSPI 200_RSI14'] = data_2008['KOSPI 200_RSI14'].replace([np.inf, -np.inf], np.nan)

# =========================
# 5. Bollinger Bands(15, 2) 생성
# =========================
bb_mid = data_2008[price_col].rolling(window=15).mean()
bb_std = data_2008[price_col].rolling(window=15).std()

data_2008['KOSPI 200_BB_MID15'] = bb_mid
data_2008['KOSPI 200_BB_UPPER15'] = bb_mid + 2 * bb_std
data_2008['KOSPI 200_BB_LOWER15'] = bb_mid - 2 * bb_std

# =========================

In [10]:
# =========================
# 6. ADX / DMI(14) 생성
#    - ADX: 추세 강도(방향성 없음)
#    - +DI/-DI: 상승/하락 방향성
#    - DMI: +DI와 -DI의 차이
# =========================
adx_period = 14
high = data_2008['KOSPI 200 High']
low = data_2008['KOSPI 200 Low']
close = data_2008[price_col]

# True Range(TR): 당일 변동폭 + 전일 종가 갭을 반영
tr_components = pd.concat([
    (high - low),
    (high - close.shift(1)).abs(),
    (low - close.shift(1)).abs()
], axis=1)
tr = tr_components.max(axis=1)

# Directional Movement(+DM, -DM)
up_move = high.diff()
down_move = -low.diff()
plus_dm = pd.Series(
    np.where((up_move > down_move) & (up_move > 0), up_move, 0.0),
    index=data_2008.index
)
minus_dm = pd.Series(
    np.where((down_move > up_move) & (down_move > 0), down_move, 0.0),
    index=data_2008.index
)

# Wilder 방식과 유사한 지수평활(alpha=1/n)
atr = tr.ewm(alpha=1 / adx_period, adjust=False).mean()
plus_di = 100 * (plus_dm.ewm(alpha=1 / adx_period, adjust=False).mean() / atr)
minus_di = 100 * (minus_dm.ewm(alpha=1 / adx_period, adjust=False).mean() / atr)

# DX -> ADX
dx = (plus_di - minus_di).abs() / (plus_di + minus_di)
dx = dx.replace([np.inf, -np.inf], np.nan) * 100
adx = dx.ewm(alpha=1 / adx_period, adjust=False).mean()

data_2008['KOSPI 200_PDI14'] = plus_di
data_2008['KOSPI 200_MDI14'] = minus_di
data_2008['KOSPI 200_DMI14'] = plus_di - minus_di
data_2008['KOSPI 200_ADX14'] = adx

# =========================
# 7. ROC(12) 생성
#    ROC = ((오늘 종가 - n일 전 종가) / n일 전 종가) * 100
# =========================
roc_period = 12
data_2008['KOSPI 200_ROC12'] = data_2008[price_col].pct_change(periods=roc_period) * 100

# =========================
# 8. 2009 데이터에 붙일 피처 목록
# =========================
feature_cols = [
    'KOSPI 200_MA5',
    'KOSPI 200_MA15',
    'KOSPI 200_EMA5',
    'KOSPI 200_EMA15',
    'KOSPI 200_RSI14',
    'KOSPI 200_BB_MID15',
    'KOSPI 200_BB_UPPER15',
    'KOSPI 200_BB_LOWER15',
    'KOSPI 200_PDI14',
    'KOSPI 200_MDI14',
    'KOSPI 200_DMI14',
    'KOSPI 200_ADX14',
    'KOSPI 200_ROC12'
]

# =========================
# 9. data_2009 인덱스 기준으로 결합
# =========================
data_2009[feature_cols] = data_2008[feature_cols].reindex(data_2009.index)

# 백업
data_tech_features = data_2009.copy()

# =========================
# 10. 확인
# =========================
print(data_tech_features.head())
print("\n결측치 개수:")
print(data_tech_features[feature_cols].isna().sum())

data_tech_features.to_csv(
    r'..\..\data\processed\data_tech_features.csv',
    index=True
)

            KODEX 200  Brent Crude Oil   Gold Spot      USD/KRW       NASDAQ  \
Date                                                                           
2009-04-17    17370.0        51.959999  867.400024  1325.800049  1673.069946   
2009-04-20    17480.0        51.959999  887.000000  1327.500000  1608.209961   
2009-04-21    17480.0        51.959999  882.099976  1354.300049  1643.849976   
2009-04-22    17715.0        51.959999  891.799988  1346.599976  1646.119995   
2009-04-23    17895.0        51.959999  905.900024  1333.599976  1652.209961   

                KOSDAQ  KOSPI 200 Close  KOSPI 200 Open  KOSPI 200 High  \
Date                                                                      
2009-04-17  483.799988       171.330002      174.029999      174.990005   
2009-04-20  491.940002       172.300003      172.550003      173.039993   
2009-04-21  497.190002       171.960007      168.580002      172.350006   
2009-04-22  509.899994       174.399994      173.279999      175

### 데이터 병합

In [11]:
data_tech=pd.read_csv(r'..\..\data\processed\data_tech_features.csv')
data_GARCH=pd.read_csv(r'..\..\data\processed\data_one_step_GARCH_GJR-GARCH_VaR_Label.csv')

#두 데이터 병합하기
merge_final_data = pd.merge(data_tech, data_GARCH, on='Date', how='inner')
merge_final_data.set_index('Date', inplace=True)

merge_final_data.to_csv(r'..\..\data\processed\data_middle.csv', index=True)

### Signal족 생성

In [12]:
import pandas as pd

# data_middle.csv 파일 불러옴
data_middle = pd.read_csv(
    r"..\..\data\processed\data_middle.csv",
    index_col=0,
    parse_dates=True
)

# Signal1: MA5와 MA15의 골든크로스와 데드크로스
data_middle['Signal1'] = 'Stay'

buy_cond = (data_middle['KOSPI 200_MA5'].shift(1) <= data_middle['KOSPI 200_MA15'].shift(1)) & (data_middle['KOSPI 200_MA5'] > data_middle['KOSPI 200_MA15'])
sell_cond = (data_middle['KOSPI 200_MA5'].shift(1) >= data_middle['KOSPI 200_MA15'].shift(1)) & (data_middle['KOSPI 200_MA5'] < data_middle['KOSPI 200_MA15'])

data_middle.loc[buy_cond, 'Signal1'] = 'Buy'
data_middle.loc[sell_cond, 'Signal1'] = 'Sell'

# Signal2: EMA5와 EMA15의 골든크로스와 데드크로스
data_middle['Signal2'] = 'Stay'

buy_cond = (data_middle['KOSPI 200_EMA5'].shift(1) <= data_middle['KOSPI 200_EMA15'].shift(1)) & (data_middle['KOSPI 200_EMA5'] > data_middle['KOSPI 200_EMA15'])
sell_cond = (data_middle['KOSPI 200_EMA5'].shift(1) >= data_middle['KOSPI 200_EMA15'].shift(1)) & (data_middle['KOSPI 200_EMA5'] < data_middle['KOSPI 200_EMA15'])

data_middle.loc[buy_cond, 'Signal2'] = 'Buy'
data_middle.loc[sell_cond, 'Signal2'] = 'Sell'

# Signal3: RSI14의 과매도와 과매수 신호에 따른 매매 신호
data_middle['Signal3'] = 'Stay'

buy_cond = (data_middle['KOSPI 200_RSI14'].shift(1) <= 30) & (data_middle['KOSPI 200_RSI14'] > 30)
sell_cond = (data_middle['KOSPI 200_RSI14'].shift(1) >= 70) & (data_middle['KOSPI 200_RSI14'] < 70)

data_middle.loc[buy_cond, 'Signal3'] = 'Buy'
data_middle.loc[sell_cond, 'Signal3'] = 'Sell'

# Signal4: Bollinger Bands 기반의 매매 신호
data_middle['Signal4'] = 'Stay'

buy_cond = (data_middle['KOSPI 200 Close'].shift(1) <= data_middle['KOSPI 200_BB_LOWER15'].shift(1)) & (data_middle['KOSPI 200 Close'] > data_middle['KOSPI 200_BB_LOWER15'])
sell_cond = (data_middle['KOSPI 200 Close'].shift(1) >= data_middle['KOSPI 200_BB_UPPER15'].shift(1)) & (data_middle['KOSPI 200 Close'] < data_middle['KOSPI 200_BB_UPPER15'])

data_middle.loc[buy_cond, 'Signal4'] = 'Buy'
data_middle.loc[sell_cond, 'Signal4'] = 'Sell'

### Signal족 가변수화

In [13]:
signal_cols = ['Signal1', 'Signal2', 'Signal3', 'Signal4']

# 1. 각 Signal의 범주 순서를 지정
for col in signal_cols:
    data_middle[col] = pd.Categorical(
        data_middle[col],
        categories=['Stay', 'Buy', 'Sell']
    )

# 2. Stay를 기준범주로 하여 가변수 생성
signal_dummies = pd.get_dummies(
    data_middle[signal_cols],
    prefix=signal_cols,
    prefix_sep='_',
    drop_first=True,
    dtype=int
)

# 3. 원본 데이터프레임에 붙이기
data_middle = pd.concat([data_middle, signal_dummies], axis=1)

data_middle = data_middle.drop(columns=['Signal1', 'Signal2', 'Signal3', 'Signal4'])

cols = [col for col in data_middle.columns if col != 'Risk_Label'] + ['Risk_Label']
data_middle = data_middle[cols]

# KOSPI 200 Return과 return(%)가 중복이므로 컬럼 제거
data_middle=data_middle.drop(columns=['KOSPI 200 Return'])

### 컬럼 재배치

In [14]:
new_order = [
    'KODEX 200', 'NASDAQ', 'KOSDAQ',
    'Brent Crude Oil', 'Gold Spot',
    'USD/KRW',
    'KOSPI 200 Close', 'KOSPI 200 Open', 'KOSPI 200 High', 'KOSPI 200 Low','KOSPI 200 Volume',
    'return(%)','KOSPI 200 lagged_1_return(%)', 'KOSPI 200 lagged_2_return(%)',
    'VKOSPI_Close','VKOSPI_Change(%)',
    'KOSPI 200_MA5', 'KOSPI 200_MA15', 'KOSPI 200_EMA5', 'KOSPI 200_EMA15',
    'KOSPI 200_RSI14', 'KOSPI 200_BB_MID15', 'KOSPI 200_BB_UPPER15', 'KOSPI 200_BB_LOWER15',
    'KOSPI 200_PDI14', 'KOSPI 200_MDI14', 'KOSPI 200_DMI14','KOSPI 200_ADX14', 'KOSPI 200_ROC12',
    'GARCH_mu_t1', 'GARCH_sigma_t1', 'GJR_sigma_t1', 'GARCH_VaR_5_t1', 'GJR_VaR_5_t1',
    'Signal1_Buy', 'Signal1_Sell', 'Signal2_Buy', 'Signal2_Sell',
    'Signal3_Buy', 'Signal3_Sell', 'Signal4_Buy', 'Signal4_Sell',
    'Risk_Label'
]

# 혹시 누락/오타 체크
missing_cols = [col for col in new_order if col not in data_middle.columns]
extra_cols = [col for col in data_middle.columns if col not in new_order]

print("누락된 컬럼:", missing_cols)
print("추가 컬럼:", extra_cols)

# 재배치
data_middle = data_middle[new_order]

누락된 컬럼: []
추가 컬럼: []


## Boruta 이전 최종 데이터 추출

In [15]:
# 데이터 저장
data_middle.to_csv("../../data/processed/data_final.csv", index=True)

#data_middel label칼럼 shift(-1)하고 마지막 행 drop하기
data_middle_shift = data_middle.copy()
data_middle_shift['Risk_Label'] = data_middle_shift['Risk_Label'].shift(-1)
data_middle_shift = data_middle_shift.dropna(subset=['Risk_Label'])

data_middle_shift.to_csv("../../data/processed/data_final_shift.csv", index=True)